In [1]:
######################################
### attempting a WordNet converter ###
##### to convert to WN LMF 1.3 ? #####
######################################

# we have:
# CSV/TSV data: Vietnamese, Mongolian
# XML data: Hungarian
# others like Turkish, Korean, that should be ok not converted

In [2]:
from xml.sax.handler import feature_external_ges

#########################
### imports and stuff ###
#########################

### pip install bs4
### pip install lxml

### for xml files: ElementTree? ###
from pathlib import Path

import requests
import lxml
import xml.etree.ElementTree as ET
from bs4 import BeautifulSoup, element


In [3]:
def show_xml(element, level=0, count=100):
    c = 0
    indent = "  " * level
    text = (element.text or "").strip()
    attrs = element.attrib

    if attrs:
        attr_str = ", ".join([f"{k}={v}" for k, v in attrs.items()])
        print(f"{indent}- {element.tag} [{attr_str}]")
    else:
        print(f"{indent}- {element.tag}: {text}")

    for child in element:
        show_xml(child, level + 1)

        if c >= count:
            break

        c += 1

In [6]:
def xml_structure(element, level=0, seen=None):
    if seen is None:
        seen = set()

    indent = "  " * level
    tag = element.tag
    attrs = tuple(sorted(element.attrib.keys()))

    sig = (tag, attrs)
    if sig not in seen:
        seen.add(sig)
        if attrs:
            print(f"{indent}- {tag} [{', '.join(attrs)}]")
        else:
            print(f"{indent}- {tag}")

    for child in element:
        xml_structure(child, level + 1, seen)

In [7]:
def xml_id(element, lit):

    if element.find('ID') is not None and element.find('SYNONYM').find('LITERAL') is not None:
        lit[element.find('ID').text] = element.find('SYNONYM').find('LITERAL').text


In [8]:
def id_dict(path, lit=None, count=100, encoding="utf-8"):

    if lit is None:
        lit = {}

    tree = ET.parse(path)
    root = tree.getroot()

    for elem in root.findall('SYNSET'):
        xml_id(elem, lit)

    return lit

In [9]:
### trying to join the two upper functions, to make it easier
### works, the dictionaries are identical

### so this function just creates a dict with ID: literal
### NL is a label to mark non-lexicality, these entries can be excluded

def xml_id_dict(path, lit=None):

    if lit is None:
        lit = {}

    tree = ET.parse(path)
    root = tree.getroot()

    for elem in root.findall('SYNSET'):
        if elem.find('NL') is None:
            lit[elem.find('ID').text] = []
            for l in elem.find('SYNONYM').findall('LITERAL'):
                lit[elem.find('ID').text].append(l.text)

    return lit

In [10]:
### trying to join the two upper functions, to make it easier
### works, the dictionaries are identical

### so this function just creates a dict with literal: ID, ALL the literals

def xml_lit_dict(path, lit=None):

    if lit is None:
        lit = {}

    tree = ET.parse(path)
    root = tree.getroot()

    for elem in root.findall('SYNSET'):
        if elem.find('NL') is None:
            for l in elem.find('SYNONYM').findall('LITERAL'):
                if l not in lit:
                    lit[l.text] = []
                    lit[l.text].append(elem.find('ID').text)
                else: lit[l.text].append(elem.find('ID').text)
        else:
            print(f'{elem.find("ID").text}: {elem.find("SYNONYM").find("LITERAL").text}')
    return lit

In [5]:
#hunDict = id_dict(r'C:/Users/torto/Desktop/Studium/BA/code/rand_files/HuWN/HuWN_final4.xml')
#hunDict2 = xml_lit_dict(r'C:/Users/torto/Desktop/Studium/BA/code/rand_files/HuWN/HuWN_final4.xml')
#print(hunDict2.get('HuWN-747815818-n'))
print(xml_structure(r'C:/Users/E T/Desktop/Studium/BA/code/wn_analysis/code/WordNets/HuWN/huwn.xml'))

AttributeError: 'str' object has no attribute 'tag'

In [11]:
### now we can write a function that takes in an ID/literal and returns
### relevant info (literal, hypernyms, hyponyms, synonyms
### what about multiple literals??? e.g. tüntető

def xml_getSynset(path, lit='', id=''):

    if lit == '' and id == '':
        return 'provide at least either ID or LITERAL'

    tree = ET.parse(path)
    root = tree.getroot()
    id_dct = {}

    if id == '':
        for elem in root.findall('SYNSET'):
            lits = elem.find('SYNONYM').findall('LITERAL')
            for l in lits:
                if lit == l.text:
                    if elem.find('ILR').find('TYPE').text == 'hypernym':
                        id_dct[elem.find('ID').text] = [elem.find('ILR').text]
                    else: id_dct[elem.find('ID').text] = ['']
                    id_dct[elem.find('ID').text].append(l.text)
                    id_dct[elem.find('ID').text].append(elem.find('POS').text)
                    if elem.find('ILR').find('TYPE').text == 'hyponym':
                        id_dct[elem.find('ID').text].append(elem.find('ILR').text)

    else:
        for elem in root.findall('SYNSET'):
            lits = elem.find('SYNONYM').findall('LITERAL')
            if id == elem.find('ID').text:
                if elem.find('ILR').find('TYPE').text == 'hypernym':
                    id_dct[id] = [elem.find('ILR').text]
                else: id_dct[id] = ['']
                for l in lits:
                    id_dct[id].append(l.text)
                id_dct[id].append(elem.find('POS').text)
                if elem.find('ILR').find('TYPE').text == 'hyponym':
                    id_dct[id].append(elem.find('ILR').text)

    return id_dct


In [12]:
xml_getSynset(r'C:/Users/torto/Desktop/Studium/BA/code/rand_files/HuWN/HuWN_final4.xml', id='ENG20-04398733-n')

FileNotFoundError: [Errno 2] No such file or directory: 'C:/Users/torto/Desktop/Studium/BA/code/rand_files/HuWN/HuWN_final4.xml'

In [13]:
### just quick checks to find LITERAL or ID in HunWN

tree = ET.parse(r'C:/Users/E T/Desktop/Studium/BA/code/wn_analysis/code/WordNets/HuWN/huwn.xml')
root = tree.getroot()

for e in root.findall('SYNSET'):
    for syn in e.find('SYNONYM').findall('LITERAL'):
        if 'hiba' in syn.text:
            show_xml(e)

###for e in root.findall('SYNSET'):
###    if e.find('ID').text == 'ENG20-02196975-v':
###        show_xml(e)

- SYNSET: 
  - ID: ENG20-00065349-n
  - ID3: ENG30-00070965-n
  - POS: n
  - SYNONYM: 
    - LITERAL: hiba
      - SENSE: 1
    - LITERAL: vétség
      - SENSE: 1
  - ILR: ENG20-00060780-n
    - TYPE: hypernym
  - ILR: ENG20-00066405-n
    - TYPE: hyponym
  - ILR: ENG20-00066617-n
    - TYPE: hyponym
  - ILR: ENG20-00066952-n
    - TYPE: hyponym
  - ILR: ENG20-00067911-n
    - TYPE: hyponym
  - ILR: ENG20-00068407-n
    - TYPE: hyponym
  - ILR: ENG20-00068507-n
    - TYPE: hyponym
  - ILR: ENG20-00068673-n
    - TYPE: hyponym
  - ILR: ENG20-01178066-n
    - TYPE: hyponym
  - DEF: Hibás döntés, tudatlanság vagy figyelmetlenség miatt elkövetett helytelen cselekedet.
  - BCS: 1
  - USAGE: Nagy hibát követett el.
  - STAMP: almasi 2008/03/11
  - DOMAIN: factotum
  - SUMO: SubjectiveAssessmentAttribute
    - TYPE: =
  - EKSZ: vétség_1_1
    - TYPE: =
  - EKSZ: hiba_1_1
    - TYPE: =
- SYNSET: 
  - ID: ENG20-00066212-n
  - ID3: ENG30-00072068-n
  - POS: n
  - SYNONYM: 
    - LITERAL: hiba
  

In [9]:
tree = ET.parse(r'C:/Users/torto/Desktop/Studium/BA/code/rand_files/HuWN/HuWN_final4.xml')
root = tree.getroot()

lst = root.findall('SYNSET')
lit = {}
c=0

#print(lst[0].find('SYNONYM'))
#print(lit)

for elem in lst:
    if elem.find('ID').text == 'ENG20-02362229-a':
        print(elem.find('SYNONYM').find('LITERAL').text)

FileNotFoundError: [Errno 2] No such file or directory: 'C:/Users/torto/Desktop/Studium/BA/code/rand_files/HuWN/HuWN_final4.xml'

In [10]:
tree = ET.parse(r'C:/Users/E T/Desktop/Studium/BA/code/wn_analysis/code/WordNets/HuWN/huwn.xml')
#tree = ET.parse(r'C:/Users/E T/Desktop/Studium/BA/code/wn_analysis/code/WordNets/HuWN/HuWN_final4.xml')
root = tree.getroot()

#show_xml(root.find('SYNSET'))
#print(root.findall('SYNSET'))

'''for child in root.findall('SYNSET'):
    if child.tag == 'ID':
        show_xml(child)
    break
lst = root.findall('SYNSET')
#print(lst[0].find('ID').text)
c=0
for child in lst:

    if child.find('ID') is not None:
        print(f"{child.find('ID')}: {child.find('ID').text}")
    if child.find('POS') is not None:
        print(f"{child.find('POS')}- {child.find('POS').text}")
    if child.tag == 'ILR':
        for c in child:
            print(f"{child.text}- {c.text}")
    c+=1
    if c >= 10:
        break'''

# xml_structure(root)
show_xml(root)
#xml_id(root)

- WNXML: 
  - SYNSET: 
    - ID: ENG20-08550154-n
    - POS: n
    - SYNONYM: 
      - LITERAL: Las Cruces
        - SENSE: 1
    - ILR: ENG20-08135936-n
      - TYPE: hypernym
    - ILR: ENG20-08549007-n
      - TYPE: holo_part
    - DEF: Város Új-Mexikó déli részén, a Rio Grande partján.
    - USAGE: Las Cruces-ben pihentünk egy órát.
    - SNOTE: ??
    - STAMP: szauter 2006/08/31
  - SYNSET: 
    - ID: ENG20-08172407-n
    - POS: n
    - SYNONYM: 
      - LITERAL: Durres
        - SENSE: 1
    - ILR: ENG20-08005407-n
      - TYPE: hypernym
    - ILR: ENG20-08106737-n
      - TYPE: hypernym
    - ILR: ENG20-08171998-n
      - TYPE: holo_part
    - DEF: Nyugat-albániai kikötőváros.
    - STAMP: szk 2006/09/05
  - SYNSET: 
    - ID: ENG20-01828848-a
    - POS: a
    - SYNONYM: 
      - LITERAL: hencegő
        - SENSE: 1
      - LITERAL: kérkedő
        - SENSE: 1
    - ILR: ENG20-01827430-a
      - TYPE: similar_to
    - DEF: Feltűnően, arrogánsan büszke.
    - USAGE: Sok nyúl élt az

In [14]:
tr_tree = ET.parse(r'C:\\Users\\E T\\Desktop\\Studium\\BA\\code\\wn_analysis\\code\\WordNets\\TurkishWordNet-Py\\WordNet\\data\\turkish_wordnet.xml')
#en_tree = ET.parse(r'C:\\Users\\E T\\Desktop\\Studium\\BA\\code\\wn_analysis\\code\\WordNets\\TurkishWordNet-Py\\WordNet\\data\\english_wordnet_version_31.xml')

#tr_tree = ET.parse(r'C:/Users/E T/Desktop/Studium/BA/code/wn_analysis/code/WordNets/HuWN/HuWN_final4.xml')

#en_root = en_tree.getroot()
tr_root = tr_tree.getroot()

show_xml(tr_root)
#xml_structure(root)

- SYNSETS: 
  - SYNSET: 
    - ID: TUR10-0000000
    - SYNONYM: 
      - LITERAL: (özel isim)
        - SENSE: 1
    - POS: n
    - SR: TUR10-0379560
      - TYPE: HYPERNYM
    - DEF: Bir özel isim
  - SYNSET: 
    - ID: TUR10-0000003
    - SYNONYM: 
      - LITERAL: (zaman)
        - SENSE: 1
    - POS: n
    - SR: TUR10-0869550
      - TYPE: HYPERNYM
    - DEF: Bir zaman
  - SYNSET: 
    - ID: TUR10-0000004
    - SYNONYM: 
      - LITERAL: (tarih)
        - SENSE: 1
    - POS: n
    - SR: TUR10-0747200
      - TYPE: HYPERNYM
    - DEF: Bir tarih
  - SYNSET: 
    - ID: TUR10-0000006
    - SYNONYM: 
      - LITERAL: (hashtag)
        - SENSE: 1
    - POS: n
    - SR: TUR10-0579300
      - TYPE: HYPERNYM
    - DEF: Bir hashtag
  - SYNSET: 
    - ID: TUR10-0000007
    - SYNONYM: 
      - LITERAL: (eposta)
        - SENSE: 1
    - POS: n
    - SR: TUR10-0844140
      - TYPE: HYPERNYM
    - DEF: Bir elektronik posta
  - SYNSET: 
    - ID: TUR10-0000010
    - SYNONYM: 
      - LITERAL: (tam

In [36]:
tree = ET.parse(r'C:/Users/torto/Desktop/Studium/BA/code/rand_files/omw-da-1.4/omw-da/omw-da.xml')
root = tree.getroot()

#show_xml(root)
xml_structure(root)

- LexicalResource
  - Lexicon [citation, email, id, label, language, license, url, version]
    - Requires [id, version]
    - LexicalEntry [id]
      - Lemma [partOfSpeech, writtenForm]
      - Sense [id, synset]
    - Synset [id, ili, members, partOfSpeech]


In [9]:
def create_WN_from_xml(xml, wnfile):
    if Path(xml).exists():
        try:
            tree = ET.parse(r''+xml)
        except ET.ParseError:
            print(f"{xml} is not a valid XML file")

        root = ET.Element("LexicalResource")

        e1 = ET.SubElement(root, "Lexicon")
        e1.set("language", "hn")
        e1.set("url", "https://rgai.inf.u-szeged.hu/file/136")

        e2 = ET.SubElement(e1, "Requires")
        e2.set('id','1')
        e2.set('version','1')

        e3 = ET.SubElement(e1, "LexicalEntry")
        e3.set("id", "SOME_ID")

        e4 = ET.SubElement(e3, "Lemma")
        e4.set("partOfSpeech", "POS")
        e4.set("writtenForm", "FORM")

        e4 = ET.SubElement(e3, "Sense")
        e4.set("id", "LEX_ID")
        e4.set("synset", "LEX_SYNSET")

        e2 = ET.SubElement(e1, "Synset")
        e2.set('id','SYNSET_ID')
        e2.set('ili','ILI')
        e2.set('members','LEX_ID')
        e2.set('partOfSpeech','POS')

        #for synset in tree.findall('SYNSET'):

        tree = ET.ElementTree(root)
        with open(wnfile, "wb") as f:
            tree.write(f)

    else:
        print(f"{xml} is not an existing file")

In [10]:
create_WN_from_xml('C:/Users/torto/Desktop/Studium/BA/code/rand_files/omw-da-1.4/omw-da/omw-da.xml', 'C:/Users/torto/Desktop/Studium/BA/code/rand_files/HunWN_WN.xml')

C:/Users/torto/Desktop/Studium/BA/code/rand_files/omw-da-1.4/omw-da/omw-da.xml is not an existing file
